In [ ]:
# Install/load
install.packages(c("MatchIt", "cobalt", "dplyr", "readr", "glue")) # run once
library(MatchIt)
library(cobalt)
library(dplyr)
library(readr)
library(glue)

In [ ]:
results <- "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/results/2025-12-03_viral_disease_control_cohort_creation"

results1 <- "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/results/2025-12-02_create_input_dataframes_for_matchit" 

In [ ]:
perform_matching <- function(input_data, 
                             match_ratio = 6, 
                             exact_vars = c("sex", "ancestry")) {
  
  # 1) Run the matching
  # The formula case ~ age + sex + ancestry is hard-coded as requested
  m.out <- MatchIt::matchit(
    case ~ age + sex + ancestry,
    data = input_data,
    method = "nearest",           # Use nearest neighbor...
    exact = exact_vars,           # ...but ONLY within exact groups
    ratio = match_ratio,          # Desired controls per case
    distance = "mahalanobis"
  )
  
  # 2) Print diagnostics to the console
  message("--- Matching Summary ---")
  print(summary(m.out))
  
  message("\n--- Balance Plot (Love Plot) ---")
  # Use tryCatch in case 'cobalt' isn't loaded or plot fails
  tryCatch({
    # We must explicitly 'print()' a plot when inside a function
    print(cobalt::love.plot(m.out))
  }, error = function(e) {
    message("Could not generate love.plot. Error: ", e$message)
  })
  
  # 3) Get matched/weighted cohorts
  matched_data <- MatchIt::match.data(m.out)
  
  # 4) Return the final matched data frame
  return(matched_data)
}

In [ ]:
batch_match_and_save <- function(input_folder,
                                 output_folder,
                                 file_pattern = "\\.csv$",
                                 ...) {
  # Create output folder if it doesn't exist
  dir.create(output_folder, showWarnings = FALSE, recursive = TRUE)
  
  # List input files
  files <- list.files(
    input_folder,
    pattern = file_pattern,
    full.names = TRUE
  )
  
  if (length(files) == 0) {
    message("No files found in ", input_folder,
            " matching pattern: ", file_pattern)
    return(invisible(NULL))
  }
  
  for (f in files) {
    message("Processing file: ", f)
    
    # 1) Read the input data
    input_data <- read.csv(f)
    
    # 2) Run your matching function
    # '...' lets you pass match_ratio, exact_vars, etc. if you want
    matched_data <- perform_matching(input_data, ...)
    
    # 3) Build a corresponding output filename
    base_name <- tools::file_path_sans_ext(basename(f))  # e.g. "cohort1"
    out_file  <- file.path(
      output_folder,
      paste0("matched_", base_name, ".csv")
    )
    
    # 4) Save matched dataset
    write.csv(matched_data, out_file, row.names = FALSE)
    
    message("Saved matched data to: ", out_file, "\n")
  }
  
  invisible(NULL)
}


In [ ]:
##Function calls

batch_match_and_save(
  input_folder  = glue("{results1}/ns_matchit")
  output_folder = glue("{results}/ns_matched")
)



batch_match_and_save(
  input_folder  = glue("{results1}/b1_matchit")
  output_folder = glue("{results}/b1_matched")
)


batch_match_and_save(
  input_folder  = glue("{results1}/ns_vax_matchit")
  output_folder = glue("{results}/ns_vax_matched")
)


batch_match_and_save(
  input_folder  = glue("{results1}/ns_raw_vax_matchit")
  output_folder = glue("{results}/ns_raw_vax_matched")
)



batch_match_and_save(
  input_folder  = glue("{results1}/c1_matchit")
  output_folder = glue("{results}/c1_matched")
)



batch_match_and_save(
  input_folder  = glue("{results1}/d1_matchit")
  output_folder = glue("{results}/d1_matched")
)




batch_match_and_save(
  input_folder  = glue("{results1}/flu_matchit")
  output_folder = glue("{results}/flu_matched")
)




batch_match_and_save(
  input_folder  = glue("{results1}/covid_matchit")
  output_folder = glue("{results}/covid_matched")
)



